# 6. OCR文本错漏频发？结合LLM纠错，让图像文本也能精准使用

### 1.1 错误类型分析

OCR识别错误通常可以分为以下几类：

- **字符识别错误**
- **文字遗漏**
- **多字重复**
- **格式混乱**
- **特殊符号识别错误**

这些错误往往源于以下几个方面：

- **图像质量不佳**
- **字体样式复杂**
- **背景干扰**
- **OCR算法限制**

### 1.2 传统解决方案的局限性

传统的OCR优化方法主要包括：

- **图像预处理**
- **后处理规则**
- **模型微调**

## 结合 LLM 进行文本纠错的新思路
- 充分发挥 LLM 的语言建模能力
- OCR + LLM 的协同流程

我们可以将整个OCR处理流程分为两个阶段：

1. OCR识别阶段：使用 PaddleOCR 5 对输入图像进行识别，得到初步的文本结果。
2. LLM纠错阶段：将OCR输出的文本送入大语言模型，由其进行语义级别的纠错和优化。

具体流程如下：
[图像] → [PaddleOCR 5] → [初步OCR文本] → [LLM纠错] → [最终文本]

# 实战操作

[快速安装飞桨平台工具](https://www.paddlepaddle.org.cn/install/quick?docurl=undefined)

In [1]:
! python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/


In [2]:
! python -m pip install "paddleocr[all]"

In [4]:
! paddleocr ocr -i https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/general_ocr_002.png --use_doc_orientation_classify False --use_doc_unwarping False --use_textline_orientation False 


D:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
��Ϣ: ���ṩ��ģʽ�޷��ҵ��ļ���
D:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\awu\.paddlex\official_models\PP-OCRv5_server_det`.
I0324 19:45:59.930712 13068 onednn_context.cc:81] oneDNN v3.6.2
Crea

![OCR 示例图片](https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/general_ocr_002.png  "通用 OCR 示例图像")

In [5]:
# PP-OCRv5 示例
from paddleocr import PaddleOCR
# 初始化 PaddleOCR 实例
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False)
# 对示例图像执行 OCR 推理 
result = ocr.predict(
    input="https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/general_ocr_002.png")
# 可视化结果并保存 json 结果
for res in result:
    res.print()
    res.save_to_img("output")
    res.save_to_json("output")

d:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
d:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
d:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', No

![PP-OCRv5](https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/AIStudio/application_help/demo_images/algorithm_ppocrv5.png)

![PP-ChatOCRV4](https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/AIStudio/application_help/demo_images/algorithm_ppchatocrv4.png)

In [9]:
from paddleocr import PPChatOCRv4Doc
import os
from dotenv import load_dotenv
load_dotenv()

BCE_API_KEY = os.getenv("BCE_API_KEY")

chat_bot_config = {
    "module_name": "chat_bot",
    "model_name": "ernie-4.0-8k",
    "base_url": "https://qianfan.baidubce.com/v2",
    "api_type": "openai",
    "api_key": BCE_API_KEY
}

retriever_config = {
    "module_name": "retriever",
    "model_name": "embedding-v1",
    "base_url": "https://qianfan.baidubce.com/v2",
    "api_type": "qianfan",
    "api_key": BCE_API_KEY
}

pipeline = PPChatOCRv4Doc(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False
)

visual_predict_res = pipeline.visual_predict(
    input="https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/vehicle_certificate-1.png",
    use_seal_recognition=True,
    use_table_recognition=True,
)

visual_info_list = []
for res in visual_predict_res:
    visual_info_list.append(res["visual_info"])

vector_info = pipeline.build_vector(
    visual_info=visual_info_list, flag_save_bytes_vector=True, retriever_config=retriever_config
)

chat_result = pipeline.chat(
    key_list = ["该车辆是哪个公司生产的"],
    visual_info=visual_info_list,
    vector_info=vector_info,
    mllm_predict_info=None,
    chat_bot_config=chat_bot_config,
    retriever_config=retriever_config
)

print(f"Chat Result: {chat_result}")

Creating model: ('RT-DETR-H_layout_3cls', None)
Using official model (RT-DETR-H_layout_3cls), the model files will be automatically downloaded and saved in `C:\Users\awu\.paddlex\official_models\RT-DETR-H_layout_3cls`.
Fetching 6 files: 100%|██████████| 6/6 [00:18<00:00,  3.04s/it]
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `C:\Users\awu\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Fetching 6 files: 100%|██████████| 6/6 [00:02<00:00,  2.79it/s]
Creating model: ('PP-OCRv4_server_det', None)
Using official model (PP-OCRv4_server_det), the model files will be automatically downloaded and saved in `C:\Users\awu\.paddlex\official_models\PP-OCRv4_server_det`.
Fetching 6 files: 100%|██████████| 6/6 [00:05<00:00,  1.07it/s]
Creating model: ('PP-OCRv4_server_rec_doc', None)
Using official model (PP-OCRv4_server_rec_doc), the model files will be automatically downlo

Chat Result: {'chat_res': {'该车辆是哪个公司生产的': '东风柳州汽车有限公司'}}


![ocr_result](https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/vehicle_certificate-1.png)